<a href="https://colab.research.google.com/github/tojotk/ict_assesment/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# NLP - Natural Language Processing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 36.1 MB/s eta 0:00:00


# Libraries

In [ ]:
import numpy as np
import pandas as pd
import string
import nltk

from nltk.stem import PorterStemmer , WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import AdaBoostClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

import gensim.downloader as api
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
from gensim.models import Word2Vec # Import the Word2Vec class

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Read Dataset

In [ ]:
filepath = '/content/drive/MyDrive/AI ML Course/Data/spam.xlsx'
df_spam = pd.read_excel(filepath)
df_spam

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will �_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


# EDA

In [ ]:
df_spam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [ ]:
# relevant data present only in first 2 columns
df_spam = df_spam[['v1', 'v2']]
df_spam.head(3)

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...


In [ ]:
# check unique values count
df_spam['v1'].value_counts()

,count
v1,
ham,4825
spam,747


# Preprocessing

## Punctuations

In [ ]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
sample_text = "Hello! Can we test #1 ($50.99) item? Yes, @John_Doe said: 'It's a well-known state-of-the-art 50% / 100% match [id: #789]—or is it?' Check example.com {file: data.csv, status: active | pending} <ok?> ~all done^;"

In [ ]:
sample_text

"Hello! Can we test #1 ($50.99) item? Yes, @John_Doe said: 'It's a well-known state-of-the-art 50% / 100% match [id: #789]—or is it?' Check example.com {file: data.csv, status: active | pending} <ok?> ~all done^;"

In [ ]:
def remove_punctuation(text):
  punctuationless_text =''.join([i for i in text if i not in string.punctuation])
  # comparing each character in 'text' against the punctuation list
  return punctuationless_text

In [ ]:
print('before')
print('************************************')
print(sample_text)
print('\n,\n')
print('after')
print('************************************')
print(remove_punctuation(sample_text))

before
************************************
Hello! Can we test #1 ($50.99) item? Yes, @John_Doe said: 'It's a well-known state-of-the-art 50% / 100% match [id: #789]—or is it?' Check example.com {file: data.csv, status: active | pending} <ok?> ~all done^;

,

after
************************************
Hello Can we test 1 5099 item Yes JohnDoe said Its a wellknown stateoftheart 50  100 match id 789—or is it Check examplecom file datacsv status active  pending ok all done


## Lowercasing

In [ ]:
punctuation_free_text = remove_punctuation(sample_text)

lower_case_text = punctuation_free_text.lower()
print('before')
print('************************************')
print(punctuation_free_text)
print('\n,\n')
print('after')
print('************************************')
print(lower_case_text)

before
************************************
Hello Can we test 1 5099 item Yes JohnDoe said Its a wellknown stateoftheart 50  100 match id 789—or is it Check examplecom file datacsv status active  pending ok all done

,

after
************************************
hello can we test 1 5099 item yes johndoe said its a wellknown stateoftheart 50  100 match id 789—or is it check examplecom file datacsv status active  pending ok all done


## Tokenaziation

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# we convert the entire text into a list of unique words
# function to get tokens from the sentence
# input should be a text after punctuation removed and in lowercase
def tokenization(text):
    words_list = nltk.word_tokenize(text)
    return words_list

In [ ]:
tokens = tokenization(lower_case_text)
tokens

['hello',
 'can',
 'we',
 'test',
 '1',
 '5099',
 'item',
 'yes',
 'johndoe',
 'said',
 'its',
 'a',
 'wellknown',
 'stateoftheart',
 '50',
 '100',
 'match',
 'id',
 '789—or',
 'is',
 'it',
 'check',
 'examplecom',
 'file',
 'datacsv',
 'status',
 'active',
 'pending',
 'ok',
 'all',
 'done']

## STop words removal

In [ ]:
nltk.download('stopwords')
# get all the identified words in english
stop_words_list = nltk.corpus.stopwords.words('english')
stop_words_list

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [ ]:
def stop_remove(tokens_list_arg):
  stop_word_removed_text = [i for i in tokens_list_arg if i not in stop_words_list]
  return stop_word_removed_text

In [ ]:
print('before')
print('**********')
print('count of tokens =', len(tokens))
print('/n/n')
print('after')
clean_tokens = stop_remove(tokens)
print('count of tokens = ',  len(clean_tokens))
print('**********')
print(clean_tokens)

before
**********
count of tokens = 31
/n/n
after
count of tokens =  24
**********
['hello', 'test', '1', '5099', 'item', 'yes', 'johndoe', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'status', 'active', 'pending', 'ok', 'done']


## Stemming

In [ ]:
stem_obj = PorterStemmer()
def stemming(clean_tokens_arg):
  stem_list = [stem_obj.stem(word) for word in clean_tokens_arg ]
  return stem_list

In [ ]:
filtered_tokens = stop_remove(tokens)
stem_list = stemming(filtered_tokens)
print(filtered_tokens)
print(len(filtered_tokens))

print(stem_list)
print(len(stem_list))

['hello', 'test', '1', '5099', 'item', 'yes', 'johndoe', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'status', 'active', 'pending', 'ok', 'done']
24
['hello', 'test', '1', '5099', 'item', 'ye', 'johndo', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'statu', 'activ', 'pend', 'ok', 'done']
24


## Lemmatization

In [ ]:
lema_obj = WordNetLemmatizer()
def lemmatization(stem_token_list_arg):
  lemma_list = [lema_obj.lemmatize(word) for word in stem_token_list_arg]
  return lemma_list

In [ ]:
nltk.download('wordnet')
lemma_list = lemmatization(stem_list)
print(lemma_list)
print(len(lemma_list))

[nltk_data] Downloading package wordnet to /root/nltk_data...


['hello', 'test', '1', '5099', 'item', 'ye', 'johndo', 'said', 'wellknown', 'stateoftheart', '50', '100', 'match', 'id', '789—or', 'check', 'examplecom', 'file', 'datacsv', 'statu', 'activ', 'pend', 'ok', 'done']
24


### Applying Preprocessing to DataFrame



In [ ]:
df_spam['v2'] = df_spam['v2'].astype(str)

# Punctuation Removal
punctuationless_texts = []
for text in df_spam['v2']:
    punctuationless_texts.append(remove_punctuation(text))
df_spam['punctuationless_text'] = punctuationless_texts

# Lowercasing
lowercase_texts = []
for text in df_spam['punctuationless_text']:
    lowercase_texts.append(text.lower())
df_spam['lowercase_text'] = lowercase_texts

# Tokenization
tokenized_texts = []
for text in df_spam['lowercase_text']:
    tokenized_texts.append(tokenization(text))
df_spam['tokens'] = tokenized_texts

# Stopword Removal
stopwords_removed_lists = []
for token_list in df_spam['tokens']:
    stopwords_removed_lists.append(stop_remove(token_list))
df_spam['removed_stopwords'] = stopwords_removed_lists

# Stemming
stemmed_token_lists = []
for token_list in df_spam['removed_stopwords']:
    stemmed_token_lists.append(stemming(token_list))
df_spam['stemmed_tokens'] = stemmed_token_lists

# Lemmatization
lemmatized_token_lists = []
for token_list in df_spam['stemmed_tokens']:
    lemmatized_token_lists.append(lemmatization(token_list))
df_spam['lemmatized_tokens'] = lemmatized_token_lists

display(df_spam.head())

/tmp/ipykernel_1558/2675383456.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_spam['v2'] = df_spam['v2'].astype(str)
/tmp/ipykernel_1558/2675383456.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_spam['punctuationless_text'] = punctuationless_texts
/tmp/ipykernel_1558/2675383456.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.o

,v1,v2,punctuationless_text,lowercase_text,tokens,removed_stopwords,stemmed_tokens,lemmatized_tokens
0,ham,"Go until jurong point, crazy.. Available only ...",Go until jurong point crazy Available only in ...,go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o...","[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,Ok lar... Joking wif u oni...,Ok lar Joking wif u oni,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]","[ok, lar, joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[ok, lar, joke, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin...","[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,ham,U dun say so early hor... U c already then say...,U dun say so early hor U c already then say,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t...","[u, dun, say, early, hor, u, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]","[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...",Nah I dont think he goes to usf he lives aroun...,nah i dont think he goes to usf he lives aroun...,"[nah, i, dont, think, he, goes, to, usf, he, l...","[nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho...","[nah, dont, think, goe, usf, live, around, tho..."


In [ ]:
display(df_spam[['v1', 'v2', 'lemmatized_tokens']].head())

,v1,v2,lemmatized_tokens
0,ham,"Go until jurong point, crazy.. Available only ...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,Ok lar... Joking wif u oni...,"[ok, lar, joke, wif, u, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,ham,U dun say so early hor... U c already then say...,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...","[nah, dont, think, goe, usf, live, around, tho..."


In [ ]:
df_spam['clean_text'] = df_spam['lemmatized_tokens'].apply(lambda x: ' '.join(x))
display(df_spam[['v1', 'v2', 'clean_text']].head())

,v1,v2,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,ham,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah dont think goe usf live around though


### Bag Of Words




In [ ]:
count_vectorizer_obj = CountVectorizer()
count_vec = count_vectorizer_obj.fit_transform(df_spam['clean_text'])
count_vec.shape

(5572, 7971)

## Classification


### Label Encoding

In [ ]:
label_encoder_model = LabelEncoder()
df_spam['v1_encoded'] = label_encoder_model.fit_transform(df_spam['v1'])

In [ ]:
X = count_vec
y = df_spam['v1_encoded']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
LinearRegression_model = LogisticRegression()
LinearRegression_model.fit(X_train, y_train)

y_pred = LinearRegression_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.979372197309417

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       1.00      0.85      0.92       150

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115


Confusion Matrix:
[[965   0]
 [ 23 127]]


### Boosting

In [ ]:
 # Initialize AdaBoostClassifier
adaboost_model = AdaBoostClassifier(random_state=42)

# Train the model
adaboost_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_adaboost = adaboost_model.predict(X_test)

# Evaluate the model
print(f"AdaBoost Accuracy: {accuracy_score(y_test, y_pred_adaboost)}")
print("\nAdaBoost Classification Report:")
print(classification_report(y_test, y_pred_adaboost))
print("\nAdaBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_adaboost))

AdaBoost Accuracy: 0.9255605381165919

AdaBoost Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       965
           1       0.95      0.47      0.63       150

    accuracy                           0.93      1115
   macro avg       0.94      0.73      0.79      1115
weighted avg       0.93      0.93      0.91      1115


AdaBoost Confusion Matrix:
[[961   4]
 [ 79  71]]


## TF - IDF

In [ ]:
corpus = df_spam['clean_text'].tolist()
corpus

['go jurong point crazi avail bugi n great world la e buffet cine got amor wat',
 'ok lar joke wif u oni',
 'free entri 2 wkli comp win fa cup final tkt 21st may 2005 text fa 87121 receiv entri questionstd txt ratetc appli 08452810075over18',
 'u dun say earli hor u c alreadi say',
 'nah dont think goe usf live around though',
 'freemsg hey darl 3 week word back id like fun still tb ok xxx std chg send �150 rcv',
 'even brother like speak treat like aid patent',
 'per request mell mell oru minnaminungint nurungu vettam set callertun caller press 9 copi friend callertun',
 'winner valu network custom select receivea �900 prize reward claim call 09061701461 claim code kl341 valid 12 hour',
 'mobil 11 month u r entitl updat latest colour mobil camera free call mobil updat co free 08002986030',
 'im gon na home soon dont want talk stuff anymor tonight k ive cri enough today',
 'six chanc win cash 100 20000 pound txt csh11 send 87575 cost 150pday 6day 16 tsandc appli repli hl 4 info',
 'urg

In [ ]:
TF_IDF_vectorizer = TfidfVectorizer()
tf_idf_vector = TF_IDF_vectorizer.fit_transform(corpus)
tf_idf_vector.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
TF_IDF_vectorizer.get_feature_names_out()

array(['008704050406', '0089mi', '0121', ..., 'zoom', 'zouk', 'zyada'],
      dtype=object)

## N-grams

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
# n_gram range parameter set to make sure we are getting bigrams only
bigram_vectors = bigram_vectorizer.fit_transform(corpus)
bigram_vectors.toarray()


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
bigram_vectorizer.get_feature_names_out()


array(['008704050406 sp', '0089mi last', '0121 2025050', ..., 'zoom cine',
       'zouk nichol', 'zyada kisi'], dtype=object)

## Word to Vector

In [ ]:
api.info()['models'].keys() # word2 Vec model avaliabble in gensim library

dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])

In [ ]:
# loading word2vec-google-news-300 model
word2vecmodel = api.load('word2vec-google-news-300')
print('model loading successful')

[==================================================] 100.0% 1662.8/1662.8MB downloaded
model loading successful


In [ ]:
word2vecmodel['king']

array([ 1.25976562e-01,  2.97851562e-02,  8.60595703e-03,  1.39648438e-01,
       -2.56347656e-02, -3.61328125e-02,  1.11816406e-01, -1.98242188e-01,
        5.12695312e-02,  3.63281250e-01, -2.42187500e-01, -3.02734375e-01,
       -1.77734375e-01, -2.49023438e-02, -1.67968750e-01, -1.69921875e-01,
        3.46679688e-02,  5.21850586e-03,  4.63867188e-02,  1.28906250e-01,
        1.36718750e-01,  1.12792969e-01,  5.95703125e-02,  1.36718750e-01,
        1.01074219e-01, -1.76757812e-01, -2.51953125e-01,  5.98144531e-02,
        3.41796875e-01, -3.11279297e-02,  1.04492188e-01,  6.17675781e-02,
        1.24511719e-01,  4.00390625e-01, -3.22265625e-01,  8.39843750e-02,
        3.90625000e-02,  5.85937500e-03,  7.03125000e-02,  1.72851562e-01,
        1.38671875e-01, -2.31445312e-01,  2.83203125e-01,  1.42578125e-01,
        3.41796875e-01, -2.39257812e-02, -1.09863281e-01,  3.32031250e-02,
       -5.46875000e-02,  1.53198242e-02, -1.62109375e-01,  1.58203125e-01,
       -2.59765625e-01,  

In [ ]:
# testing words in the model
word2vecmodel['Queen']

array([-0.22070312, -0.17480469, -0.10498047,  0.2578125 ,  0.16210938,
       -0.13085938, -0.16699219,  0.07373047, -0.07226562,  0.02404785,
       -0.13964844,  0.02197266,  0.17675781, -0.19140625,  0.0378418 ,
       -0.01782227, -0.03710938, -0.03735352,  0.15625   ,  0.08837891,
        0.0534668 , -0.02392578, -0.2734375 , -0.2578125 , -0.00720215,
        0.06933594, -0.21777344, -0.10058594,  0.2421875 ,  0.03417969,
       -0.12890625, -0.1171875 , -0.18261719,  0.04321289, -0.125     ,
       -0.09960938,  0.26367188,  0.375     , -0.32421875, -0.1328125 ,
       -0.13378906, -0.50390625, -0.05908203,  0.04077148,  0.23730469,
       -0.03393555, -0.01495361, -0.09765625, -0.06445312,  0.02087402,
       -0.10302734,  0.10449219,  0.20019531, -0.16503906, -0.01196289,
        0.30859375, -0.41015625, -0.22070312,  0.08056641, -0.12792969,
        0.13085938,  0.28515625, -0.07275391,  0.02612305,  0.01916504,
       -0.16992188,  0.01745605,  0.13085938, -0.17089844, -0.10

In [ ]:
# find cosine similarity
similarity = word2vecmodel.similarity('king', 'queen')
similarity


np.float32(0.6510957)

In [ ]:
# find cosine similarity
similarity = word2vecmodel.similarity('woman', 'man')
similarity


np.float32(0.76640123)

In [ ]:
word2vecmodel.most_similar('king', topn= 10)


[('kings', 0.7138045430183411),
 ('queen', 0.6510956883430481),
 ('monarch', 0.6413194537162781),
 ('crown_prince', 0.6204220056533813),
 ('prince', 0.6159993410110474),
 ('sultan', 0.5864824056625366),
 ('ruler', 0.5797567367553711),
 ('princes', 0.5646552443504333),
 ('Prince_Paras', 0.5432944297790527),
 ('throne', 0.5422105193138123)]

## Continuous Bag of Words

In [ ]:
corpus_1 = ["King and Queen rule the kingdom",
            "The man is walking in the park",
            "The woman is cooking food in the kitchen",
            "The child is playing with toys",
            "Computer and laptop are electronic devices",
            "Python and Java are popular programming languages"]

In [ ]:
# tokenize the entire corpus

In [ ]:
tokenized_corpus = [word_tokenize(document.lower()) for document in corpus_1]
tokenized_corpus

[['king', 'and', 'queen', 'rule', 'the', 'kingdom'],
 ['the', 'man', 'is', 'walking', 'in', 'the', 'park'],
 ['the', 'woman', 'is', 'cooking', 'food', 'in', 'the', 'kitchen'],
 ['the', 'child', 'is', 'playing', 'with', 'toys'],
 ['computer', 'and', 'laptop', 'are', 'electronic', 'devices'],
 ['python', 'and', 'java', 'are', 'popular', 'programming', 'languages']]

In [ ]:
# building CBoW neural network
model_cbow = Word2Vec(sentences = tokenized_corpus,
                      vector_size = 50, # word embedding size
                      window = 5,  # context information window
                      epochs = 20,
                      min_count = 1 # ignore words with frequency < 1
                      )

In [ ]:
model_cbow.wv.most_similar('king', topn= 3)

[('programming', 0.2684629261493683),
 ('python', 0.2641042470932007),
 ('is', 0.22139960527420044)]